# 🚗 Car Market Trends Analysis — Notebook 2
## Exploratory Data Analysis (EDA)

**Project:** Car Market Trends Analysis with Car Dekho Data  
**Input:** Cleaned dataset (299 rows, 11 columns)  
**Goal:** Discover patterns, distributions, and relationships in the used-car data.

---
### Analyses Covered
| # | Analysis |
|---|----------|
| 1-2 | Basic counts and unique models |
| 3 | Selling price distribution |
| 4 | Present price distribution |
| 5 | Manufacturing year distribution |
| 6 | Car age vs selling price |
| 7 | Kilometres driven vs selling price |
| 8 | Average price by fuel type |
| 9 | Average price by seller type |
| 10 | Average price by transmission |
| 11 | Average price by owner count |
| 12 | Top car models by listing frequency |
| 13 | Correlation heatmap |

---
## Setup — Import Libraries & Load Data

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
pd.set_option('display.float_format', '{:.2f}'.format)

# Load the cleaned CSV saved by Notebook 01
CLEANED_PATH = os.path.join('..', 'data', 'car_dekho_cleaned.csv')

# Fallback: load and clean on the fly if cleaned file not found
if not os.path.exists(CLEANED_PATH):
    import sys
    sys.path.insert(0, os.path.join('..', 'src'))
    from data_cleaning import get_clean_data
    df = get_clean_data()
    print('Loaded via data_cleaning module.')
else:
    df = pd.read_csv(CLEANED_PATH)
    print(f'Loaded cleaned CSV. Shape: {df.shape}')

df.head(3)

---
## Analysis 1 & 2 — Basic Counts

In [ ]:
print(f'Total listings        : {len(df)}')
print(f'Unique car/bike names : {df["Car_Name"].nunique()}')
print(f'Year range            : {df["Year"].min()} – {df["Year"].max()}')
print(f'Selling price range   : Rs {df["Selling_Price"].min():.2f}L – Rs {df["Selling_Price"].max():.2f}L')
print()

print('Fuel Type breakdown:')
print(df['Fuel_Type'].value_counts())
print()

print('Seller Type breakdown:')
print(df['Seller_Type'].value_counts())
print()

print('Transmission breakdown:')
print(df['Transmission'].value_counts())

---
## Analysis 3 — Selling Price Distribution

In [ ]:
print('=== Selling Price Stats ===')
print(f'Mean   : Rs {df["Selling_Price"].mean():.2f}L')
print(f'Median : Rs {df["Selling_Price"].median():.2f}L')
print(f'Std    : Rs {df["Selling_Price"].std():.2f}L')
print()
print('Finding: Mean > Median → right-skewed distribution.')
print('         A few expensive cars pull the average up.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: full distribution
sns.histplot(df['Selling_Price'], bins=35, kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('Selling Price Distribution (All)', fontsize=13)
axes[0].set_xlabel('Selling Price (Rs Lakhs)')
axes[0].set_ylabel('Number of Listings')

# Right: zoom ≤ Rs 15L for main cluster
under15 = df[df['Selling_Price'] <= 15]
sns.histplot(under15['Selling_Price'], bins=30, kde=True, color='mediumpurple', ax=axes[1])
axes[1].set_title('Selling Price ≤ Rs 15L (Main Cluster)', fontsize=13)
axes[1].set_xlabel('Selling Price (Rs Lakhs)')
axes[1].set_ylabel('Number of Listings')

plt.suptitle('Analysis 3 — Selling Price Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Analysis 4 — Present Price Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(df['Present_Price'], bins=35, kde=True, color='darkorange', ax=ax)
ax.set_title('Present Price Distribution', fontsize=13)
ax.set_xlabel('Present Price (Rs Lakhs)')
ax.set_ylabel('Number of Listings')
plt.tight_layout()
plt.show()

print(f'Note: Max present price = Rs {df["Present_Price"].max():.1f}L (Land Cruiser 2010)')

---
## Analysis 5 — Manufacturing Year Distribution

In [ ]:
yr_c = df['Year'].value_counts().sort_index()
print('Listings per year:')
print(yr_c.to_string())

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(yr_c.index.astype(str), yr_c.values, color='steelblue', edgecolor='white')
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.3, str(int(h)),
            ha='center', va='bottom', fontsize=9)
ax.set_title('Listings by Manufacture Year', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Finding: Most listings are from 2012–2017.')

---
## Analysis 6 — Car Age vs Selling Price

In [ ]:
corr6 = df['Car_Age'].corr(df['Selling_Price'])
print(f'Pearson r (Car_Age vs Selling_Price) = {corr6:.4f}')
print('Negative r → older cars tend to have lower selling prices.')

fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=df, x='Car_Age', y='Selling_Price',
                hue='Fuel_Type', alpha=0.65, s=60, palette='Set1', ax=ax)

# Trend line
z = np.polyfit(df['Car_Age'], df['Selling_Price'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['Car_Age'].min(), df['Car_Age'].max(), 100)
ax.plot(x_line, p(x_line), color='black', linewidth=1.8,
        linestyle='--', label='Trend line')
ax.legend(title='Fuel Type', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.set_title(f'Car Age vs Selling Price  (r = {corr6:.2f})', fontsize=13)
ax.set_xlabel('Car Age (Years)')
ax.set_ylabel('Selling Price (Rs Lakhs)')
plt.tight_layout()
plt.show()

---
## Analysis 7 — Kilometres Driven vs Selling Price

In [ ]:
corr7 = df['Kms_Driven'].corr(df['Selling_Price'])
print(f'Pearson r (Kms_Driven vs Selling_Price) = {corr7:.4f}')
print('Near-zero r because the dataset mixes cars and 2-wheelers.')

fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=df, x='Kms_Driven', y='Selling_Price',
                hue='Fuel_Type', alpha=0.55, s=55, palette='Set2', ax=ax)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_title(f'Km Driven vs Selling Price  (r = {corr7:.2f})', fontsize=13)
ax.set_xlabel('Kilometres Driven')
ax.set_ylabel('Selling Price (Rs Lakhs)')
ax.legend(title='Fuel Type', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

---
## Analysis 8 — Average Selling Price by Fuel Type

In [ ]:
fuel_summary = (
    df.groupby('Fuel_Type')['Selling_Price']
    .agg(Count='count', Mean='mean', Median='median', Std='std')
    .round(2)
    .sort_values('Mean', ascending=False)
)
print(fuel_summary)
print('\nNote: CNG only has 2 records — do not draw strong conclusions from it.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
order = fuel_summary.index.tolist()

sns.barplot(data=df, x='Fuel_Type', y='Selling_Price', order=order,
            estimator='mean', palette='Set2', errorbar='sd', ax=axes[0])
axes[0].set_title('Mean Selling Price by Fuel Type')
axes[0].set_ylabel('Avg Price (Rs Lakhs)')

sns.boxplot(data=df, x='Fuel_Type', y='Selling_Price', order=order,
            palette='Set2', ax=axes[1])
axes[1].set_title('Price Distribution by Fuel Type')
axes[1].set_ylabel('Selling Price (Rs Lakhs)')

plt.suptitle('Analysis 8 — Selling Price by Fuel Type', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Analysis 9 — Average Selling Price by Seller Type

In [ ]:
seller_summary = (
    df.groupby('Seller_Type')['Selling_Price']
    .agg(Count='count', Mean='mean', Median='median')
    .round(2)
)
print(seller_summary)
print('\nNote: Price gap reflects vehicle type mix — individuals mostly sell 2-wheelers.')

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
sns.barplot(data=df, x='Seller_Type', y='Selling_Price',
            estimator='mean', palette='pastel', errorbar='sd', ax=axes[0])
axes[0].set_title('Mean Selling Price by Seller Type')
axes[0].set_ylabel('Avg Price (Rs Lakhs)')

sns.boxplot(data=df, x='Seller_Type', y='Selling_Price',
            palette='pastel', ax=axes[1])
axes[1].set_title('Price Distribution by Seller Type')
axes[1].set_ylabel('Selling Price (Rs Lakhs)')

plt.tight_layout()
plt.show()

---
## Analysis 10 — Average Selling Price by Transmission

In [ ]:
trans_summary = (
    df.groupby('Transmission')['Selling_Price']
    .agg(Count='count', Mean='mean', Median='median')
    .round(2)
)
print(trans_summary)
print('\nFinding: Automatic cars average Rs 5L more than Manual.')

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
sns.barplot(data=df, x='Transmission', y='Selling_Price',
            estimator='mean', palette=['steelblue', 'coral'],
            errorbar='sd', ax=axes[0])
axes[0].set_title('Mean Selling Price by Transmission')
axes[0].set_ylabel('Avg Price (Rs Lakhs)')

sns.violinplot(data=df, x='Transmission', y='Selling_Price',
               palette=['steelblue', 'coral'], inner='quartile', ax=axes[1])
axes[1].set_title('Price Distribution — Violin')
axes[1].set_ylabel('Selling Price (Rs Lakhs)')

plt.tight_layout()
plt.show()

---
## Analysis 11 — Average Selling Price by Owner Count

In [ ]:
own_summary = (
    df.groupby('Owner')['Selling_Price']
    .agg(Count='count', Mean='mean', Median='median')
    .round(2)
)
print(own_summary)
print('\nCaution: Owner=3 has only 1 record — not statistically meaningful.')

avg_own = df.groupby('Owner')['Selling_Price'].mean().reset_index()
cnt_map = df.groupby('Owner').size().to_dict()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(avg_own['Owner'].astype(str), avg_own['Selling_Price'],
              color=['steelblue','mediumpurple','coral','seagreen'])
for bar, (_, row) in zip(bars, avg_own.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'n={cnt_map[int(row["Owner"])]}', ha='center', fontsize=9)
ax.set_title('Avg Selling Price by Number of Previous Owners')
ax.set_xlabel('Previous Owners')
ax.set_ylabel('Avg Price (Rs Lakhs)')
plt.tight_layout()
plt.show()

---
## Analysis 12 — Top Car Names by Listing Frequency

In [ ]:
top15 = df['Car_Name'].value_counts().head(15).sort_values()
print('Top 15 most listed cars/bikes:')
print(top15.to_string())

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['steelblue' if i >= 12 else 'lightsteelblue' for i in range(len(top15))]
top15.plot(kind='barh', color=colors, ax=ax)
for i, v in enumerate(top15.values):
    ax.text(v + 0.1, i, str(v), va='center', fontsize=9)
ax.set_title('Top 15 Most Listed Cars / Bikes', fontsize=13)
ax.set_xlabel('Number of Listings')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

---
## Analysis 13 — Correlation Heatmap

In [ ]:
num_cols = ['Selling_Price', 'Present_Price', 'Kms_Driven',
            'Year', 'Car_Age', 'Owner', 'Price_Depreciation']
corr = df[num_cols].corr().round(3)

print('Correlation with Selling_Price (sorted by absolute value):')
sp = corr['Selling_Price'].drop('Selling_Price').sort_values(key=abs, ascending=False)
for feat, val in sp.items():
    bar = '█' * int(abs(val) * 20)
    direction = '+' if val > 0 else '-'
    print(f'  {feat:<22} {direction}{abs(val):.3f}  {bar}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Correlation Heatmap — Numeric Features', fontsize=13, pad=14)
plt.tight_layout()
plt.show()

print('Key finding: Present_Price is the strongest predictor of Selling_Price (r = 0.876).')

---
## EDA Summary

| Finding | Value |
|---------|-------|
| Most listed model | Honda City (26 listings) |
| Diesel avg price | Rs 10.10L vs Petrol Rs 3.26L |
| Automatic avg price | Rs 9.07L vs Manual Rs 3.92L |
| Dealer avg price | Rs 6.63L vs Individual Rs 0.87L |
| Strongest predictor | Present_Price (r = 0.876) |
| Car age correlation | r = -0.234 (older = cheaper) |
| Kms correlation | r = 0.029 (very weak) |

> **Next step:** Open `03_visualizations.ipynb` to see all static charts saved as PNG files.